# Background Theory
## Preprocessing Pipelines for Machine Learning — Penguin Species Project

This notebook covers the *concepts* behind every tool used in the project:
`ColumnTransformer`, `OneHotEncoder`, `SimpleImputer`, `StandardScaler`, `Pipeline`,
train/test splitting, cross-validation, and classification metrics.

Read this before (or alongside) the skeleton notebook. No code needs to be run here —
it's meant to build intuition.

## 1. Why Preprocess Data at All?

Real-world datasets like `penguins.csv` mix:
- **Numeric columns** (bill length, flipper length, body mass) — different scales and units.
- **Categorical columns** (`sex`, `island`) — text labels, not numbers.
- **Missing values** — sensors fail, forms are left blank, etc.

Most scikit-learn models (Logistic Regression, KNN, SVM, etc.) require:
1. All input to be **numeric**.
2. **No missing values**.
3. Features on **comparable scales** (for distance- or gradient-based models).

So before we can fit *any* model, we need a preprocessing stage that turns messy,
mixed-type, incomplete data into a clean numeric matrix.

## 2. Encoding Categorical Variables — `OneHotEncoder`

A column like `sex` has values `Male` / `Female`. A model can't multiply weights by
text, so we need a numeric representation.

**One-Hot Encoding** creates one binary (0/1) column per category:

| sex    | sex_Female | sex_Male |
|--------|-----------|----------|
| Male   | 0         | 1        |
| Female | 1         | 0        |

This avoids implying a false *order* between categories (unlike `LabelEncoder`,
which would turn `Male`/`Female` into `0`/`1` as if one were "greater" than the other).

For `island` (`Biscoe`, `Dream`, `Torgersen`), the same idea creates 3 binary columns.

**Trade-off:** more categories → more columns (the "curse of dimensionality" in a
mild form). For low-cardinality columns like these, it's the standard choice.

## 3. Handling Missing Values — `SimpleImputer`

Rows sometimes have missing values (`NaN`). Dropping every row with *any* missing
value can throw away useful data. Instead, we **impute** (fill in) a reasonable
substitute value.

Common strategies (`SimpleImputer(strategy=...)`):

| Strategy         | What it fills in                          | Best for                    |
|------------------|--------------------------------------------|------------------------------|
| `'mean'`         | Column average                             | Numeric, normally distributed |
| `'median'`       | Column median                              | Numeric, skewed / outliers   |
| `'most_frequent'`| Most common value (mode)                   | Categorical **or** numeric   |
| `'constant'`     | A fixed value you specify (`fill_value=`)  | When missingness is meaningful |

In our project we use `'most_frequent'` because after one-hot encoding, *all*
columns (including the encoded categorical ones) are numeric, and `'most_frequent'`
is the only strategy that works safely on both encoded binary columns and
remaining numeric columns without assuming a distribution.

## 4. Feature Scaling — `StandardScaler`

Numeric features can live on very different scales:
- `bill_length_mm` ≈ 30–60
- `body_mass_g` ≈ 2700–6300

Algorithms that rely on distances or gradients (KNN, SVM, logistic regression with
regularization, neural nets) treat large-magnitude features as "more important"
purely because of their scale — not because they're actually more predictive.

**Standardization** rescales each feature to have mean 0 and standard deviation 1:

```
z = (x - mean) / std
```

After scaling, every feature contributes on equal footing. Tree-based models
(Random Forest, Gradient Boosting) don't need this, but it never hurts them either.

## 5. Doing It Column-by-Column — `ColumnTransformer`

We don't want to one-hot-encode *every* column (numeric columns don't need it),
and we don't want to scale the raw text columns. We need to apply **different
transformations to different columns of the same DataFrame, in one step**.

`ColumnTransformer` (or its shorthand `make_column_transformer`) does exactly this:

```python
ct = make_column_transformer(
    (OneHotEncoder(), ['sex', 'island']),
    remainder='passthrough'   # leave all other columns unchanged, don't drop them
)
```

- The tuple `(transformer, columns)` says "apply this transformer to these columns."
- `remainder='passthrough'` means "everything else goes through untouched."
  (The default, `remainder='drop'`, would silently delete every column not listed!)

**Important ordering effect:** the output of a `ColumnTransformer` places the
*transformed* columns first, then the passthrough columns, in the order the
transformers were listed — not the original column order.

## 6. Chaining Steps — `Pipeline`

A `Pipeline` (or `make_pipeline`) chains multiple steps so they act as a single
estimator:

```python
pipe = make_pipeline(ct, SimpleImputer(strategy='most_frequent'), StandardScaler())
```

Why not just call each step manually? Two big reasons:

1. **No data leakage.** When you call `pipe.fit(X_train)`, every step (imputer
   means, scaler means/std, encoder categories) is learned *only* from the
   training data. Calling `pipe.transform(X_test)` reuses those learned values
   instead of recomputing statistics from the test set — which would leak
   information about the test set into your preprocessing.
2. **Convenience & reproducibility.** One `.fit()` / `.predict()` call replaces
   a long manual sequence, and the whole pipeline can be cross-validated,
   grid-searched, or pickled as a single object.

`make_pipeline` auto-names each step (e.g. `columntransformer`,
`simpleimputer`, `standardscaler`); the explicit `Pipeline([...])` constructor
lets you assign custom names.

## 7. Splitting Data — Train/Test Split & Cross-Validation

If we fit a model and then measure accuracy on the *same* data it was trained on,
we generally get an overly optimistic — sometimes drastically overfit — result.

**Train/test split**: hold out a portion of the data (e.g. 20–30%) that the model
never sees during training, then evaluate on it. This estimates how well the
model will generalize to new data.

**Cross-validation (k-fold)**: split the data into *k* folds; train on *k-1* folds
and validate on the remaining fold, rotating which fold is held out, then average
the scores. This uses the data more efficiently than a single split and gives a
more robust estimate (with a spread/variance), especially useful on small
datasets like this penguins dataset (~340 rows).

## 8. Classification & Evaluation Metrics

Our target `species` has 3 classes (`Adelie`, `Chinstrap`, `Gentoo`) — a
**multi-class classification** problem.

Two simple, effective baseline classifiers:
- **K-Nearest Neighbors (KNN):** classify a point by majority vote of its *k*
  closest neighbors in feature space. Sensitive to feature scale — this is
  exactly why `StandardScaler` matters.
- **Logistic Regression:** learns a linear decision boundary (extended to
  multi-class via one-vs-rest or multinomial softmax).

**Metrics:**
- **Accuracy** — fraction of correct predictions. Fine when classes are
  roughly balanced (they are, for penguins).
- **Precision / Recall / F1** (per class) — useful when some classes matter
  more, or are rarer, than others.
- **Confusion matrix** — a table of predicted vs. actual classes; shows
  *which* species get confused with each other.

## 9. Putting It All Together — The Full Flow

```
raw DataFrame (mixed types, some missing values)
        │
        ▼
ColumnTransformer
   ├── OneHotEncoder on ['sex', 'island']
   └── passthrough on numeric columns
        │
        ▼
SimpleImputer(strategy='most_frequent')   # fill any remaining NaNs
        │
        ▼
StandardScaler                             # zero mean, unit variance
        │
        ▼
(optional) Classifier — e.g. KNeighborsClassifier
        │
        ▼
Predictions / evaluation metrics
```

Everything above the classifier is *preprocessing*; wrapping it all — including
the classifier — in one `Pipeline` means `pipe.fit(X_train, y_train)` and
`pipe.predict(X_test)` handle the entire flow safely and reproducibly.

Continue to the **skeleton notebook** to build this yourself, or the
**cheat sheet notebook** for quick syntax reference.